# FYP experimental automatic speed estimation

This CPU notebook consumes the completed private tracking and 3D-geometry experiments.
It fits a separate local road mapping for each carriageway from predicted vehicle ground footprints.
Approximate car-size priors establish a provisional scale: nominal length 4.5 m and width 1.8 m.
Sensitivity scenarios vary length from 3.2–5.2 m and width from 1.45–2.0 m, and vary the calibration track subset.

**All km/h values are experimental and unvalidated. Ranges are sensitivity envelopes, not confidence intervals.**
Predicted corners, approximate dimensions, road slope and lens distortion can introduce errors outside those ranges.
Only track segments passing geometric and temporal consistency screens receive speed labels.
These screens are not accuracy validation. Some tracks and road regions will have no speed.

This is an independent prototype using learned 3D footprints and dimension priors. It is not a reproduction
of Sochor et al.'s exact vehicle-model alignment, which requires make/model recognition and corresponding geometry.

References: [Sochor et al. 2017](https://arxiv.org/abs/1702.06451),
[UrbanOmniDetect — Turkcan, Gumaste and Kostic, CVPRW 2026](https://github.com/mkturkcan/urbanomnidetect).


In [ ]:
from pathlib import Path
import os, json, subprocess, sys
from IPython.display import display, Image, Video
import pandas as pd
input_root=Path('/kaggle/input')
def unique(name):
    paths=list(input_root.rglob(name))
    assert len(paths)==1,(name,paths)
    return paths[0]
geometry=unique('geometry_run_report.json').parent
geometry_report=json.loads((geometry/'geometry_run_report.json').read_text())
assert geometry_report['model_sha256']=='9aa55280a5ef1c9d89c0acb5baeb974b90fbdf3c1deead1413b24d7af249226d', 'Unexpected geometry model version'
os.environ.update({'FYP_SPEED_ROOT':'/kaggle/working','FYP_GEOMETRY_DIR':str(geometry),
    'FYP_SPEED_OUTPUT':'/kaggle/working/speed_experiment',
    'FYP_BASELINE_TRACKS':str(unique('tracks.csv')),
    'FYP_SPEED_SOURCE':str(unique('A3_ch14_20260822105146_20260822110347.mp4'))})
print('Input geometry:',geometry_report['model_filename'])
print('No GPU inference is repeated by this notebook.')


In [ ]:
sources={'calibrate_speed.py': '"""Experimental local road rectification from learned cuboids and explicit car-size priors.\n\nIndependent implementation of rectangle/aspect/rigidity fitting, inspired by\nSochor et al. (2017) and the UrbanOmniDetect (2026) ground-footprint approach.\nRanges are sensitivity envelopes, not confidence intervals or accuracy claims.\n"""\nfrom pathlib import Path\nfrom collections import defaultdict\nimport csv\nimport json\nimport math\nimport os\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\n\nROOT = Path(os.environ.get(\'FYP_SPEED_ROOT\',str(Path(__file__).resolve().parent)))\ndownloaded_geometry = ROOT / \'outputs\' / \'geometry\'\ndefault_geometry = downloaded_geometry if downloaded_geometry.exists() else ROOT / \'output\' / \'inputs\' / \'geometry\'\nGEOMETRY = Path(os.environ.get(\'FYP_GEOMETRY_DIR\', str(default_geometry)))\nOUT = Path(os.environ.get(\'FYP_SPEED_OUTPUT\',str(ROOT / \'outputs\' / \'speed_experiment\')))\nLENGTH_M, WIDTH_M = 4.5, 1.8\nDIMENSION_SCENARIOS = [(4.5,1.8),(3.2,1.45),(3.2,2.0),(5.2,1.45),(5.2,2.0)]\n\n\ndef project(points, H):\n    p = np.asarray(points, dtype=float)\n    flat = p.reshape(-1,2)\n    result = np.c_[flat,np.ones(len(flat))] @ H.T\n    denominator = result[:,2]\n    safe = np.where(np.abs(denominator)<1e-8, np.where(denominator>=0,1e-8,-1e-8),denominator)\n    return (result[:,:2]/safe[:,None]).reshape(p.shape)\n\n\ndef compose(theta, normalizer):\n    p,q,a,b = theta\n    return np.array([[np.exp(a),b,0],[0,1,0],[p,q,1]]) @ normalizer\n\n\ndef measures(quads):\n    edges = np.roll(quads,-1,axis=1)-quads\n    sizes = np.linalg.norm(edges,axis=2)+1e-9\n    unit = edges/sizes[:,:,None]\n    cosines = np.sum(unit*np.roll(unit,-1,axis=1),axis=2)\n    opposite = unit[:,:2,0]*unit[:,2:,1]-unit[:,:2,1]*unit[:,2:,0]\n    lengths=(sizes[:,0]+sizes[:,2])/2\n    widths=(sizes[:,1]+sizes[:,3])/2\n    return cosines,opposite,lengths,widths\n\n\ndef fit_mapping(records, width, height, length_m=LENGTH_M, width_m=WIDTH_M, fast=False):\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    ids=np.array([r[\'track_id\'] for r in records])\n    unique=np.unique(ids)\n    normalizer=np.array([[2/width,0,-1],[0,2/width,-height/width],[0,0,1]])\n    weights=np.array([1/math.sqrt(np.sum(ids==i)) for i in ids])\n    weights/=np.mean(weights)\n    groups=[np.flatnonzero(ids==i) for i in unique]\n\n    def residual(theta):\n        H=compose(theta,normalizer)\n        world=project(quads,H)\n        angles,parallel,lengths,widths=measures(world)\n        rigidity=np.zeros(len(records))\n        for group in groups:\n            rigidity[group]=np.log(lengths[group]/np.median(lengths[group]))\n        shape=np.log((lengths/widths)/(length_m/width_m))\n        geometric=np.c_[angles,parallel*0.5,shape*0.65,rigidity*0.5]*weights[:,None]\n        den=(np.c_[quads.reshape(-1,2),np.ones(len(quads)*4)]@H.T)[:,2]\n        pole=np.maximum(0,0.08-den)*15\n        return np.r_[geometric.ravel(),pole]\n\n    starts=[(0,q,a,0) for q in ([0,1,2] if not fast else [1,2]) for a in ([0,0.7] if not fast else [0])]\n    fits=[]\n    for theta in starts:\n        result=least_squares(residual,theta,bounds=([-4,-4,-2.5,-4],[4,4,2.5,4]),\n                             loss=\'soft_l1\',f_scale=0.12,max_nfev=250)\n        fits.append((float(np.mean(residual(result.x)**2)),result))\n    _,result=min(fits,key=lambda item:item[0])\n    H=compose(result.x,normalizer)\n    world=project(quads,H)\n    _,_,lengths,_=measures(world)\n    # Give each track one vote; correlated video frames do not count as independent cars.\n    scale=length_m/np.median([np.median(lengths[g]) for g in groups])\n    H=np.diag([scale,scale,1])@H\n    world=project(quads,H)\n    directions=world[:,1]-world[:,0]\n    directions/=np.linalg.norm(directions,axis=1)[:,None]\n    directions[directions[:,1]<0]*=-1\n    direction=np.median(directions,axis=0)\n    angle=math.pi/2-math.atan2(direction[1],direction[0])\n    rotation=np.array([[math.cos(angle),-math.sin(angle),0],[math.sin(angle),math.cos(angle),0],[0,0,1]])\n    H=rotation@H\n    center=np.median(project(quads,H).mean(axis=1),axis=0)\n    H=np.array([[1,0,-center[0]],[0,1,-center[1]],[0,0,1]])@H\n    H/=np.linalg.norm(H[2,:2]) if np.linalg.norm(H[2,:2])>0 else H[2,2]\n    angle_error,_,lengths,widths=measures(project(quads,H))\n    singular=np.linalg.svd(result.jac,compute_uv=False)\n    diagnostics={\'fit_track_ids\':[int(i) for i in unique],\'fit_observations\':len(records),\n       \'median_angle_error_deg\':float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angle_error),0,1))))),\n       \'median_length_m\':float(np.median(lengths)),\'median_width_m\':float(np.median(widths)),\n       \'parameter_condition_number\':float(singular[0]/max(singular[-1],1e-12)),\n       \'optimizer_success\':bool(result.success),\'theta\':result.x.tolist(),\n       \'dimension_prior_length_m\':length_m,\'dimension_prior_width_m\':width_m}\n    return H,diagnostics\n\n\ndef prepare(observations, indices, width, height):\n    by_id=defaultdict(list)\n    for r in observations:by_id[r[\'track_id\']].append(r)\n    records=[]\n    for tid,group in by_id.items():\n        group.sort(key=lambda r:r[\'time_s\'])\n        centers=np.array([[(r[\'baseline_box\'][0]+r[\'baseline_box\'][2])/2,\n                           (r[\'baseline_box\'][1]+r[\'baseline_box\'][3])/2] for r in group])\n        if len(group)<8 or np.linalg.norm(centers[-1]-centers[0])<80:continue\n        side=\'left\' if np.median(centers[:,0])<width/2 else \'right\'\n        for j,r in enumerate(group):\n            p=np.array(r[\'keypoints\'])[indices]\n            if p.shape[1]>2 and np.min(p[:,2])<0.30:continue\n            if r[\'geometry_confidence\']<0.25 or r[\'association_iou\']<0.40:continue\n            quad=p[:,:2]\n            if not np.isfinite(quad).all():continue\n            quad=quad[np.argsort(np.arctan2(*(quad-quad.mean(axis=0))[:,::-1].T))]\n            if not cv2.isContourConvex(quad.astype(np.float32)):continue\n            if abs(cv2.contourArea(quad.astype(np.float32)))<300:continue\n            if np.min(np.linalg.norm(np.roll(quad,-1,axis=0)-quad,axis=1))<8:continue\n            cx,cy=quad.mean(axis=0)\n            if not (0.20*height<cy<0.82*height and 0.02*width<cx<0.98*width):continue\n            if np.min(quad[:,0])<4 or np.max(quad[:,0])>width-4 or np.max(quad[:,1])>height-4:continue\n            lo=max(0,j-3);hi=min(len(group)-1,j+3)\n            motion=centers[hi]-centers[lo]\n            if np.linalg.norm(motion)<5:continue\n            edge=np.roll(quad,-1,axis=0)-quad\n            alignment=np.abs(edge@motion)/(np.linalg.norm(edge,axis=1)*np.linalg.norm(motion)+1e-9)\n            scores=[np.mean(alignment[[0,2]]),np.mean(alignment[[1,3]])]\n            if abs(scores[0]-scores[1])<0.08:continue\n            if scores[1]>scores[0]:quad=np.roll(quad,-1,axis=0)\n            records.append({\'track_id\':int(tid),\'time_s\':r[\'time_s\'],\'frame_index\':r[\'frame_index\'],\n                            \'side\':side,\'quad\':quad.tolist()})\n    return records\n\n\ndef estimate_track(records,H):\n    times=np.array([r[\'time_s\'] for r in records],dtype=float)\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    world=project(quads,H)\n    centers=world.mean(axis=1)\n    t=times-times.mean()\n    initial=np.r_[np.polyfit(t,centers[:,0],1),np.polyfit(t,centers[:,1],1)]\n    def residual(p):return np.r_[(p[0]*t+p[1]-centers[:,0]),(p[2]*t+p[3]-centers[:,1])]\n    fitted=least_squares(residual,initial,loss=\'soft_l1\',f_scale=0.4)\n    speed=float(np.hypot(fitted.x[0],fitted.x[2])*3.6)\n    rms=float(np.sqrt(np.mean(residual(fitted.x)**2)))\n    angles,_,lengths,widths=measures(world)\n    length_cv=float(1.4826*np.median(np.abs(lengths-np.median(lengths)))/np.median(lengths))\n    angle_error=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n    midpoint=(times[0]+times[-1])/2\n    half_speeds=[]\n    for mask in [times<=midpoint,times>midpoint]:\n        if mask.sum()>=4:\n            velocity=np.array([np.polyfit(times[mask]-times[mask].mean(),centers[mask,k],1)[0] for k in [0,1]])\n            half_speeds.append(float(np.linalg.norm(velocity)*3.6))\n    drift=abs(half_speeds[0]-half_speeds[1])/max(speed,1) if len(half_speeds)==2 else None\n    return {\'experimental_kmh\':speed,\'trajectory_fit_rms_m\':rms,\'length_relative_mad\':length_cv,\n            \'angle_error_deg\':angle_error,\'half_span_speed_relative_difference\':drift,\n            \'median_inferred_length_m\':float(np.median(lengths)),\'median_inferred_width_m\':float(np.median(widths))}\n\n\ndef sample_balanced(records, maximum=10):\n    groups=defaultdict(list)\n    for r in records:groups[r[\'track_id\']].append(r)\n    return [group[i] for group in groups.values() for i in np.unique(np.linspace(0,len(group)-1,min(len(group),maximum)).astype(int))]\n\n\ndef fit_road_mappings(records, report):\n    width,height=report[\'width\'],report[\'height\']\n    calibration={}\n    for side in [\'left\',\'right\']:\n        local=[r for r in records if r[\'side\']==side]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        if len(ids)<3:\n            calibration[side]={\'accepted\':False,\'reason\':\'Fewer than three qualifying car tracks\',\'track_ids\':ids}\n            continue\n        # Deterministic holdout by track identity; held-out predicted boxes are consistency evidence only.\n        held_ids=ids[::3]\n        fit_ids=[i for i in ids if i not in held_ids]\n        if len(fit_ids)<3:fit_ids=ids;held_ids=[]\n        fitting=sample_balanced([r for r in local if r[\'track_id\'] in fit_ids])\n        H,diagnostics=fit_mapping(fitting,width,height)\n        alternatives=[]\n        for length_m,width_m in DIMENSION_SCENARIOS[1:]:\n            alternative,_=fit_mapping(fitting,width,height,length_m,width_m,fast=True)\n            alternatives.append(alternative)\n        rng=np.random.default_rng(42)\n        for _ in range(10):\n            selected=set(rng.choice(fit_ids,size=max(3,round(len(fit_ids)*0.75)),replace=False).tolist())\n            subset=[r for r in fitting if r[\'track_id\'] in selected]\n            alternative,_=fit_mapping(subset,width,height,fast=True)\n            alternatives.append(alternative)\n        held=[r for r in local if r[\'track_id\'] in held_ids]\n        held_angle=None\n        if held:\n            angles,_,_,_=measures(project(np.array([r[\'quad\'] for r in held]),H))\n            held_angle=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n        accepted=diagnostics[\'median_angle_error_deg\']<20 and (held_angle is None or held_angle<25)\n        accepted=accepted and report[\'ground_order_vote_agreement\']>=0.8 and diagnostics[\'parameter_condition_number\']<1e5\n        diagnostics.update({\'accepted\':bool(accepted),\'held_out_track_ids\':held_ids,\n          \'held_out_median_angle_error_deg\':held_angle,\'H_image_to_experimental_meters\':H.tolist(),\n          \'alternative_mappings\':[a.tolist() for a in alternatives],\n          \'interpretation\':\'Approximate local plane and dimension-prior mapping; no measured accuracy.\'})\n        calibration[side]=diagnostics\n        diagnostics[\'support_hull_image\'] = cv2.convexHull(\n            np.array([r[\'quad\'] for r in local], dtype=np.float32).reshape(-1, 2)).reshape(-1, 2).tolist()\n    return calibration\n\n\ndef calculate_speed_rows(records, calibration):\n    speed_rows=[]\n    for side in [\'left\',\'right\']:\n        diagnostics=calibration.get(side,{})\n        if not diagnostics.get(\'accepted\'):\n            continue\n        H=np.array(diagnostics[\'H_image_to_experimental_meters\'])\n        alternatives=[np.array(a) for a in diagnostics[\'alternative_mappings\']]\n        hull=np.array(diagnostics[\'support_hull_image\'], dtype=np.float32)\n        # Restrict reuse to the road region supported by the calibration footprints.\n        local=[r for r in records if r[\'side\']==side and all(\n            cv2.pointPolygonTest(hull, (float(p[0]),float(p[1])), True)>=-0.01 for p in r[\'quad\'])]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        for tid in ids:\n            group=sorted([r for r in local if r[\'track_id\']==tid],key=lambda r:r[\'time_s\'])\n            if len(group)<8 or group[-1][\'time_s\']-group[0][\'time_s\']<1.2:continue\n            nominal=estimate_track(group,H)\n            values=[nominal[\'experimental_kmh\']]+[estimate_track(group,a)[\'experimental_kmh\'] for a in alternatives]\n            low,high=float(min(values)),float(max(values))\n            sensitivity=(high-low)/max(nominal[\'experimental_kmh\'],1)\n            reasons=[]\n            if nominal[\'length_relative_mad\']>0.25:reasons.append(\'predicted_size_inconsistent\')\n            if nominal[\'trajectory_fit_rms_m\']>0.8:reasons.append(\'trajectory_not_consistent_with_local_plane\')\n            if nominal[\'angle_error_deg\']>25:reasons.append(\'footprint_not_rectangular_after_mapping\')\n            if nominal[\'half_span_speed_relative_difference\'] is not None and nominal[\'half_span_speed_relative_difference\']>0.35:reasons.append(\'speed_changes_or_geometry_drift_across_span\')\n            if sensitivity>0.8:reasons.append(\'high_calibration_sensitivity\')\n            speed_rows.append({\'track_id\':tid,\'side\':side,\'time_start_s\':group[0][\'time_s\'],\n              \'time_end_s\':group[-1][\'time_s\'],\'geometry_observations\':len(group),\n              \'quality_passed\':not bool(reasons),\'quality_reasons\':\';\'.join(reasons),\n              \'dimension_and_fit_sensitivity_low_kmh\':low,\'dimension_and_fit_sensitivity_high_kmh\':high,\n              **nominal})\n    return speed_rows\n\n\ndef run(calibration=None, geometry_dir=None, output_dir=None, calibration_reused=None):\n    geometry_dir=Path(geometry_dir) if geometry_dir is not None else GEOMETRY\n    output_dir=Path(output_dir) if output_dir is not None else OUT\n    output_dir.mkdir(parents=True,exist_ok=True)\n    report=json.loads((geometry_dir/\'geometry_run_report.json\').read_text())\n    observations=json.loads((geometry_dir/\'observations.json\').read_text())\n    records=prepare(observations,report[\'ground_indices\'],report[\'width\'],report[\'height\'])\n    reused=(calibration is not None) if calibration_reused is None else calibration_reused\n    if calibration is None:\n        calibration=fit_road_mappings(records, report)\n    before_support_filter=len(records)\n    records=[r for r in records if calibration.get(r[\'side\'],{}).get(\'accepted\') and all(\n        cv2.pointPolygonTest(np.array(calibration[r[\'side\']][\'support_hull_image\'],dtype=np.float32),\n                             (float(p[0]),float(p[1])),True)>=-0.01 for p in r[\'quad\'])]\n    speed_rows=calculate_speed_rows(records, calibration)\n    (output_dir/\'filtered_ground_footprints.json\').write_text(json.dumps(records),encoding=\'utf-8\')\n    (output_dir/\'calibration.json\').write_text(json.dumps(calibration,indent=2),encoding=\'utf-8\')\n    fields=[\'track_id\',\'side\',\'time_start_s\',\'time_end_s\',\'geometry_observations\',\n            \'quality_passed\',\'quality_reasons\',\'dimension_and_fit_sensitivity_low_kmh\',\n            \'dimension_and_fit_sensitivity_high_kmh\',\'experimental_kmh\',\'trajectory_fit_rms_m\',\n            \'length_relative_mad\',\'angle_error_deg\',\'half_span_speed_relative_difference\',\n            \'median_inferred_length_m\',\'median_inferred_width_m\']\n    for name, rows in [(\'experimental_speeds.csv\',speed_rows),\n                       (\'qualified_experimental_speeds.csv\',[r for r in speed_rows if r[\'quality_passed\']])]:\n        with (output_dir/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n    api_results=[{\'track_id\':r[\'track_id\'],\'carriageway\':r[\'side\'],\n      \'time_interval_s\':[r[\'time_start_s\'],r[\'time_end_s\']],\n      \'experimental_kmh\':r[\'experimental_kmh\'],\n      \'sensitivity_range_kmh\':[r[\'dimension_and_fit_sensitivity_low_kmh\'],r[\'dimension_and_fit_sensitivity_high_kmh\']],\n      \'range_type\':\'dimension_and_calibration_sensitivity\',\'accuracy_validated\':False,\n      \'method\':\'learned_3d_ground_footprints_with_assumed_vehicle_dimensions\'}\n      for r in speed_rows if r[\'quality_passed\']]\n    (output_dir/\'vehicle_speed_estimates.json\').write_text(json.dumps(api_results,indent=2),encoding=\'utf-8\')\n    result={\'calibration_reused\':reused,\'source_geometry_observations\':len(observations),\'filtered_ground_footprints\':len(records),\n      \'footprints_outside_usable_calibration\':before_support_filter-len(records),\n      \'geometry_model_filename\':report[\'model_filename\'],\'geometry_model_sha256\':report[\'model_sha256\'],\n      \'ground_corner_indices\':report[\'ground_indices\'],\n      \'experimental_track_estimates\':len(speed_rows),\'quality_passed_track_estimates\':sum(r[\'quality_passed\'] for r in speed_rows),\n      \'nominal_car_length_m\':LENGTH_M,\'nominal_car_width_m\':WIDTH_M,\'dimension_scenarios_m\':DIMENSION_SCENARIOS,\n      \'range_type\':\'Dimension-prior and track-subset sensitivity envelope. Not a statistical confidence interval.\',\n      \'scope\':\'Local visible road regions on each carriageway; passenger-car hypotheses only.\',\n      \'accuracy_validated\':False,\'speed_units\':\'km/h under assumed vehicle dimensions\',\n      \'limitations\':[\'No independent speed or road-distance ground truth.\',\n        \'Ground corners are model predictions; hidden corners can be wrong.\',\n        \'A local planar road and approximately constant speed over each fitted span are assumed.\',\n        \'Dimension priors are explicit analyst assumptions, not recognized make/model dimensions.\',\n        \'Sensitivity range does not bound unmodelled errors such as lens distortion or road grade.\']}\n    (output_dir/\'speed_run_report.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    print(json.dumps(result,indent=2))\n    for r in speed_rows:print(r[\'track_id\'],r[\'side\'],round(r[\'experimental_kmh\'],1),r[\'quality_passed\'],r[\'quality_reasons\'])\n    return calibration,speed_rows,records\n\n\nif __name__==\'__main__\':run()\n', 'render_speed.py': '"""Render the offline experimental speed estimates and local mapping diagnostics."""\nfrom pathlib import Path\nimport csv\nimport json\nimport subprocess\nimport os\nfrom collections import defaultdict\nimport cv2\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom calibrate_speed import ROOT, OUT, GEOMETRY, project\n\n\ndef draw_grid(frame, calibration, footprints):\n    height,width=frame.shape[:2]\n    for side,cal in calibration.items():\n        if not cal.get(\'accepted\'):continue\n        rows=[r for r in footprints if r[\'side\']==side]\n        if not rows:continue\n        H=np.array(cal[\'H_image_to_experimental_meters\'])\n        image_points=np.array([r[\'quad\'] for r in rows]).reshape(-1,2)\n        world=project(image_points,H)\n        low,high=np.percentile(world,[3,97],axis=0)\n        mask=np.zeros((height,width),np.uint8)\n        hull=cv2.convexHull(image_points.astype(np.float32)).astype(int)\n        cv2.fillConvexPoly(mask,hull,255)\n        overlay=np.zeros_like(frame)\n        inv=np.linalg.inv(H)\n        for axis in [0,1]:\n            for value in np.arange(np.floor(low[axis]/5)*5,high[axis]+5,5):\n                segment=np.array([[low[0]-5,low[1]-5],[high[0]+5,high[1]+5]])\n                segment[:,axis]=value\n                points=project(segment,inv)\n                if np.isfinite(points).all() and np.max(np.abs(points))<100000:\n                    a,b=points.astype(int);cv2.line(overlay,tuple(a),tuple(b),(60,200,255),2)\n        selected=(mask>0)&np.any(overlay>0,axis=2)\n        frame[selected]=overlay[selected]\n        cv2.polylines(frame,[hull],True,(255,200,60),2)\n    return frame\n\n\ndef main():\n    calibration=json.loads((OUT/\'calibration.json\').read_text())\n    footprints=json.loads((OUT/\'filtered_ground_footprints.json\').read_text())\n    csv_path=OUT/\'experimental_speeds.csv\'\n    speeds=list(csv.DictReader(csv_path.open(encoding=\'utf-8\'))) if csv_path.exists() else []\n    accepted={int(r[\'track_id\']):r for r in speeds if r[\'quality_passed\']==\'True\'}\n    baseline=defaultdict(list)\n    downloaded_tracks = ROOT / \'outputs\' / \'baseline\' / \'tracks.csv\'\n    default_tracks = downloaded_tracks if downloaded_tracks.exists() else ROOT / \'output\' / \'inputs\' / \'tracking\' / \'tracks.csv\'\n    with Path(os.environ.get(\'FYP_BASELINE_TRACKS\', str(default_tracks))).open(encoding=\'utf-8\') as f:\n        for row in csv.DictReader(f):baseline[int(row[\'frame_index\'])].append(row)\n    source_value = os.environ.get(\'FYP_SPEED_SOURCE\')\n    if not source_value:\n        raise SystemExit(\'Set FYP_SPEED_SOURCE to the original CCTV video path. See README.md.\')\n    source = Path(source_value)\n    capture=cv2.VideoCapture(str(source))\n    if not capture.isOpened():\n        raise SystemExit(f\'Cannot open source video: {source}\')\n    fps=capture.get(cv2.CAP_PROP_FPS)\n    width=int(capture.get(cv2.CAP_PROP_FRAME_WIDTH));height=int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))\n    start_s=float(os.environ.get(\'FYP_START_S\',150))\n    duration_s=float(os.environ.get(\'FYP_DURATION_S\',60))\n    capture.set(cv2.CAP_PROP_POS_FRAMES,round((start_s+duration_s*0.45)*fps));ok,reference=capture.read();assert ok\n    draw_grid(reference,calibration,footprints)\n    cv2.putText(reference,\'EXPERIMENTAL local mapping | 5 m grid under assumed car dimensions\',\n                (35,55),cv2.FONT_HERSHEY_SIMPLEX,1.05,(0,255,255),2)\n    cv2.imwrite(str(OUT/\'road_mapping.jpg\'),reference)\n    start,end=round(start_s*fps),round((start_s+duration_s)*fps)\n    capture.set(cv2.CAP_PROP_POS_FRAMES,start)\n    raw=OUT/\'experimental_speed_preview.avi\'\n    writer=cv2.VideoWriter(str(raw),cv2.VideoWriter_fourcc(*\'MJPG\'),fps,(1280,720));assert writer.isOpened()\n    labelled_frames=0\n    example_saved=False\n    try:\n        for index in range(start,end):\n            ok,frame=capture.read();assert ok,index\n            frame=cv2.resize(frame,(1280,720));sx,sy=1280/width,720/height\n            cv2.rectangle(frame,(0,0),(1280,66),(20,20,20),-1)\n            cv2.putText(frame,\'EXPERIMENTAL SPEED | approximate vehicle dimensions | accuracy unvalidated\',\n                        (15,25),cv2.FONT_HERSHEY_SIMPLEX,.62,(0,220,255),2)\n            cv2.putText(frame,\'Ranges show sensitivity to car size and calibration tracks; they are not confidence intervals.\',\n                        (15,51),cv2.FONT_HERSHEY_SIMPLEX,.52,(220,220,220),1)\n            has_label=False\n            for row in baseline.get(index,[]):\n                if not row[\'track_id\']:continue\n                tid=int(float(row[\'track_id\']));time_s=float(row[\'video_time_s\'])\n                speed=accepted.get(tid)\n                active=speed is not None and float(speed[\'time_start_s\'])<=time_s<=float(speed[\'time_end_s\'])\n                x1,y1,x2,y2=[int(float(row[k])*(sx if k.startswith(\'x\') else sy)) for k in [\'x1\',\'y1\',\'x2\',\'y2\']]\n                if active:\n                    label=(f\'ID {tid}: EXP {float(speed["experimental_kmh"]):.0f} \'\n                           f\'[{float(speed["dimension_and_fit_sensitivity_low_kmh"]):.0f}-\'\n                           f\'{float(speed["dimension_and_fit_sensitivity_high_kmh"]):.0f}] km/h\')\n                    color=(0,220,255);has_label=True\n                else:label=\'\';color=(170,170,170)\n                cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)\n                if not label:continue\n                tx=max(5,min(x1,1280-cv2.getTextSize(label,cv2.FONT_HERSHEY_SIMPLEX,.55,1)[0][0]-5))\n                ty=max(83,y1-8)\n                cv2.putText(frame,label,(tx,ty),cv2.FONT_HERSHEY_SIMPLEX,.55,(0,0,0),3)\n                cv2.putText(frame,label,(tx,ty),cv2.FONT_HERSHEY_SIMPLEX,.55,color,1)\n            cv2.putText(frame,f\'t={index/fps:.2f}s | Gray boxes: speed unavailable | Speeds average the qualifying span\',\n                        (15,701),cv2.FONT_HERSHEY_SIMPLEX,.55,(0,220,255),1)\n            if has_label:labelled_frames+=1\n            if has_label and not example_saved:\n                cv2.imwrite(str(OUT/\'speed_example.jpg\'),frame);example_saved=True\n            writer.write(frame)\n    finally:capture.release();writer.release()\n    preview=OUT/\'experimental_speed_preview.mp4\'\n    subprocess.run([\'ffmpeg\',\'-y\',\'-loglevel\',\'error\',\'-i\',str(raw),\'-c:v\',\'libx264\',\'-crf\',\'24\',\n                    \'-pix_fmt\',\'yuv420p\',\'-movflags\',\'+faststart\',str(preview)],check=True)\n    raw.unlink()\n\n    fig,axes=plt.subplots(1,2,figsize=(14,7))\n    for axis,side in zip(axes,[\'left\',\'right\']):\n        cal=calibration.get(side,{})\n        if cal.get(\'accepted\'):\n            H=np.array(cal[\'H_image_to_experimental_meters\'])\n            for tid in sorted(set(r[\'track_id\'] for r in footprints if r[\'side\']==side)):\n                group=sorted([r for r in footprints if r[\'side\']==side and r[\'track_id\']==tid],key=lambda r:r[\'time_s\'])\n                centers=project(np.array([r[\'quad\'] for r in group]),H).mean(axis=1)\n                axis.plot(centers[:,0],centers[:,1],\'.-\',label=f\'ID {tid}\')\n            axis.legend(fontsize=8);axis.set_aspect(\'equal\',adjustable=\'datalim\')\n        else:axis.text(.5,.5,\'Insufficient consistent geometry\',ha=\'center\',va=\'center\',transform=axis.transAxes)\n        axis.set_title(side.capitalize()+\' carriageway: approximate local plane\')\n        axis.set_xlabel(\'Assumption-based metres\');axis.set_ylabel(\'Assumption-based metres\');axis.grid(alpha=.2)\n    fig.suptitle(\'Experimental road mapping — arbitrary local origins; no independent distance validation\')\n    fig.tight_layout();fig.savefig(OUT/\'birdseye_trajectories.jpg\',dpi=150);plt.close(fig)\n    if accepted:\n        rows=list(accepted.values());nominal=np.array([float(r[\'experimental_kmh\']) for r in rows])\n        low=np.array([float(r[\'dimension_and_fit_sensitivity_low_kmh\']) for r in rows])\n        high=np.array([float(r[\'dimension_and_fit_sensitivity_high_kmh\']) for r in rows])\n        fig,axis=plt.subplots(figsize=(10,6))\n        axis.errorbar(nominal,np.arange(len(rows)),xerr=np.stack([nominal-low,high-nominal]),fmt=\'o\',capsize=5)\n        axis.set_yticks(np.arange(len(rows)),[f\'Track {r["track_id"]} ({r["side"]})\' for r in rows])\n        axis.set_xlabel(\'Experimental km/h under assumed dimensions\')\n        axis.set_title(\'Dimension and calibration-subset sensitivity envelopes\\nNot confidence intervals; speed accuracy remains unvalidated\')\n        axis.grid(axis=\'x\',alpha=.2);fig.tight_layout();fig.savefig(OUT/\'speed_sensitivity.jpg\',dpi=150);plt.close(fig)\n    rendering={\'preview_file\':preview.name,\'frames\':end-start,\'fps\':fps,\'duration_s\':(end-start)/fps,\n               \'frames_with_experimental_speed_labels\':labelled_frames,\'accepted_track_ids\':list(accepted)}\n    (OUT/\'render_report.json\').write_text(json.dumps(rendering,indent=2),encoding=\'utf-8\')\n    print(json.dumps(rendering,indent=2))\n\n\nif __name__==\'__main__\':main()\n'}
for name,source in sources.items():
    Path('/kaggle/working',name).write_text(source,encoding='utf-8')
subprocess.check_call([sys.executable,'/kaggle/working/calibrate_speed.py'])
subprocess.check_call([sys.executable,'/kaggle/working/render_speed.py'])


In [ ]:
output=Path('/kaggle/working/speed_experiment')
report=json.loads((output/'speed_run_report.json').read_text())
print(json.dumps(report,indent=2))
if (output/'experimental_speeds.csv').exists():
    speeds=pd.read_csv(output/'experimental_speeds.csv')
    display(speeds[['track_id','side','experimental_kmh','dimension_and_fit_sensitivity_low_kmh',
                    'dimension_and_fit_sensitivity_high_kmh','quality_passed','quality_reasons']])
for name in ['road_mapping.jpg','birdseye_trajectories.jpg','speed_sensitivity.jpg']:
    if (output/name).exists():display(Image(filename=str(output/name),width=1100))
display(Video(str(output/'experimental_speed_preview.mp4'),embed=True,width=1100))
